# Phase 1: Multi-View Inference Baseline (Stages 0, 1, 2, 5)

**Objective:** Run identity-preserving 3D face geometry reconstruction from portrait photos on Kaggle GPU.
1. **Stage 0:** InsightFace detection, 5-point alignment, pose estimation.
2. **Stage 1:** MICA multi-view frontality-weighted identity shape regression (300-D FLAME beta).
3. **Stage 2:** Canonical neutral base mesh normalization (psi=0, theta=0).
4. **Stage 5:** Production retopology, ARKit-52 blendshapes, 4-tier LOD chain, skeletal armature, and chiseled/heroic stylization presets.
5. **Gate Verification:** Pairwise beta divergence (||beta_a - beta_b|| > 1e-3) and neck seam pinning (Delta v = 0).

In [ ]:
# ── CELL 1: Environment & Repository Setup ──────────────────────────────────
import os
import sys
import subprocess
from pathlib import Path

print("--- Setting Up Humanoid-Face-3D Workspace ---")
if not Path("src/pipeline.py").exists():
    !git clone https://github.com/NetPranav/Humanoid-Face-3D.git /kaggle/working/Humanoid-Face-3D
    %cd /kaggle/working/Humanoid-Face-3D
else:
    !git pull origin main

if "." not in sys.path:
    sys.path.insert(0, ".")
print("Working directory:", os.getcwd())


In [ ]:
# ── CELL 2: Install GPU Dependencies & Model Frameworks ─────────────────────
!pip install insightface onnxruntime-gpu trimesh pyyaml scipy Pillow opencv-python --quiet

import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({props.total_memory / 1e9:.1f} GB VRAM)")


In [ ]:
# ── CELL 3: Discover & Mount Attached Datasets (FLAME & MICA) ───────────────
import shutil
from pathlib import Path

print("--- Locating Model Weights in /kaggle/input ---")
# 1. Locate FLAME 2020 generic_model.pkl
flame_pkl_candidates = list(Path("/kaggle/input").glob("**/generic_model.pkl")) + list(Path(".").glob("**/generic_model.pkl"))
if not flame_pkl_candidates:
    raise FileNotFoundError("generic_model.pkl not found! Please attach dataset nightshowdown/flame-model.")
flame_pkl = flame_pkl_candidates[0]
print(f"  Found FLAME Model: {flame_pkl}")

Path("data/flame_model").mkdir(parents=True, exist_ok=True)
if not Path("data/flame_model/generic_model.pkl").exists():
    shutil.copy(flame_pkl, "data/flame_model/generic_model.pkl")

# 2. Locate MICA pretrained weights (pretrained.tar or mica.tar)
mica_candidates = (
    list(Path("/kaggle/input").glob("**/pretrained.tar")) +
    list(Path("/kaggle/input").glob("**/mica.tar")) +
    list(Path(".").glob("**/pretrained.tar")) +
    list(Path(".").glob("**/mica.tar"))
)
if not mica_candidates:
    raise FileNotFoundError("MICA checkpoint not found! Please attach dataset nightshowdown/mica-pretrained.")
mica_tar = mica_candidates[0]
print(f"  Found MICA Weights: {mica_tar}")

Path("models_cache/mica").mkdir(parents=True, exist_ok=True)
if not Path("models_cache/mica/pretrained.tar").exists():
    shutil.copy(mica_tar, "models_cache/mica/pretrained.tar")
print("✅ Model assets successfully mounted and verified.")


In [ ]:
# ── CELL 4: Initialize Face Geometry Pipeline & Stage Test Portraits ────────
from src.pipeline import FaceGeoPipeline
from src.stage5_export.stylize import STYLIZATION_PRESETS

pipeline = FaceGeoPipeline(
    config_path="configs/default.yaml",
    model_dir="models_cache"
)
print("✅ FaceGeoPipeline loaded successfully.")

# Stage test portraits into data/test_subjects/
test_dir = Path("data/test_subjects")
test_dir.mkdir(parents=True, exist_ok=True)

# Auto-populate test portraits from demo inputs if available
demo_inputs = list(Path("vendor/MICA/demo/input").glob("*.*"))
if demo_inputs:
    for img_p in demo_inputs:
        subj_name = img_p.stem
        s_folder = test_dir / subj_name
        s_folder.mkdir(parents=True, exist_ok=True)
        target = s_folder / img_p.name
        if not target.exists():
            shutil.copy(img_p, target)

subject_folders = sorted([d for d in test_dir.iterdir() if d.is_dir() and any(d.glob("*.*"))])
print(f"Test subjects ready ({len(subject_folders)}): {[s.name for s in subject_folders]}")


In [ ]:
# ── CELL 5: Execute End-to-End Multi-Subject 3D Reconstruction ───────────────
import json
import numpy as np

output_base = Path("outputs/phase1_baseline")
output_base.mkdir(parents=True, exist_ok=True)

results = {}
subject_betas = {}

for s_folder in subject_folders:
    subj_name = s_folder.name
    photos = sorted([str(p) for p in s_folder.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    if not photos:
        continue

    print(f"
" + "=" * 50)
    print(f"Reconstructing Subject: {subj_name} ({len(photos)} photo(s))...")
    out_dir = output_base / subj_name
    res = pipeline.run(photos, str(out_dir))
    results[subj_name] = res

    with open(res["manifest_path"]) as f:
        manifest = json.load(f)
    subject_betas[subj_name] = np.array(manifest["beta_shape"], dtype=np.float32)
    print(f"  ✅ Neutral Mesh: {res['obj_path']}")
    print(f"  ✅ Blendshapes:  {res.get('blendshapes_path')}")
    print(f"  ✅ Armature Rig: {res.get('armature_path')}")


In [ ]:
# ── CELL 6: Apply Stage 5 Parametric Stylization (Chiseled & Heroic) ────────
import trimesh
from src.stage5_export.exporter import Stage5Exporter

print("
--- Generating Chiseled & Heroic Stylized Meshes ---")
exporter = Stage5Exporter(enable_lods=True, enable_armature=True)

for subj_name, res in results.items():
    s_out = output_base / subj_name
    mesh = trimesh.load(res["obj_path"], process=False)
    verts = mesh.vertices
    faces = mesh.faces

    for preset in ["chiseled", "heroic"]:
        preset_dir = s_out / f"stylized_{preset}"
        m = exporter.export_production_asset(
            neutral_vertices=verts,
            faces=faces,
            output_dir=preset_dir,
            stylization_params=preset,
            export_fbx=False
        )
        max_d = m["stylization"]["max_displacement"]
        print(f"  ✅ {subj_name} [{preset}]: max displacement = {max_d:.2f}mm | Saved to {preset_dir.name}/")


In [ ]:
# ── CELL 7: Phase 1 Gate Verification (Identity Divergence & Seam Pinning) ──
print("
--- Running Phase 1 Gate Verification ---")
if len(subject_betas) >= 2:
    names = list(subject_betas.keys())
    gate_passed = True
    print("
Pairwise Shape Divergence (||beta_a - beta_b||):")
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            dist = float(np.linalg.norm(subject_betas[names[i]] - subject_betas[names[j]]))
            print(f"  {names[i]} vs {names[j]}: {dist:.4f}")
            if dist < 1e-3:
                print(f"  ❌ FAIL: {names[i]} and {names[j]} collapsed to identical shape!")
                gate_passed = False
    assert gate_passed, "Gate Failure: Identities collapsed to mean face!"
    print("
✅ GATE 1 PASSED: Identities are distinct and non-degenerate.")
else:
    print("Note: Need >= 2 subjects to evaluate pairwise distance.")

print("✅ Phase 1 Multi-View Inference Baseline verification complete.")


In [ ]:
# ── CELL 8: Visual Gallery of Reconstructed 3D Head Meshes ──────────────────
import cv2
import matplotlib.pyplot as plt

n_subj = len(results)
if n_subj > 0:
    fig, axes = plt.subplots(1, min(4, n_subj), figsize=(4 * min(4, n_subj), 4))
    if n_subj == 1:
        axes = [axes]
    for ax, (subj_name, res) in zip(axes, list(results.items())[:4]):
        prev = res.get("preview_path")
        if prev and Path(prev).exists():
            img = cv2.imread(prev)[:, :, ::-1]
            ax.imshow(img)
            ax.set_title(f"{subj_name} (Neutral 3D)")
            ax.axis("off")
        else:
            ax.text(0.5, 0.5, f"{subj_name}
Reconstruction OK", ha="center", va="center")
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(output_base / "phase1_preview_grid.png", dpi=150)
    plt.show()
    print(f"Preview gallery saved to: {output_base / 'phase1_preview_grid.png'}")
